# 29 - Rational Surgery & L-Groups via CRT Reconstruction

The Wall surgery obstruction $\theta(f, b) \in L_n(\mathbb{Z}[\pi])$ is the *final* algebraic obstruction to completing a surgery problem. Computing this group exactly is hard: $L_n(\mathbb{Z}[\pi])$ involves Witt groups of quadratic forms over $\mathbb{Z}[\pi]$, a ring that is hard to work with directly.

The strategy: *divide and conquer via localization*.
1. Compute the **rational** $L$-group: $L_n(\mathbb{Q}[\pi])$ — easy (Witt groups over a field)
2. Compute the **$p$-local** $L$-groups: $L_n(\mathbb{Z}_{(p)}[\pi])$ for each prime $p$ — moderate
3. **Chinese Remainder Theorem** patches the local answers to recover the integral obstruction

This is not an approximation — it is *exact* by Hasse-Minkowski type principles for quadratic forms.

## Learning Goals
- Understand rational vs. $p$-local vs. integral $L$-groups and why the CRT works
- Use `compute_l_group_rational` → `RationalObstruction`
- Use `compute_l_group_p_local` → `PLocalObstruction`
- Reconstruct the integral obstruction with `reconstruct_integral_obstruction`
- Read a full multi-prime report with `prime_local_obstruction_report`

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Normal map $(f, b): M \to X$ | `IntersectionForm` |
| **Algebraic** | Wall obstruction in $L_n(\mathbb{Z}[\pi])$ | `compute_l_group_rational` + `compute_l_group_p_local` |
| **Result** | Integer list of obstruction coefficients | `reconstruct_integral_obstruction` |

## Formal Background

For $n = 4k$, $L_{4k}(\mathbb{Z}[\pi]) \cong W(\mathbb{Z}[\pi])$ (Witt group). The signature gives a map $\sigma: L_0 \to \mathbb{Z}$ that is an isomorphism rationally ($L_0(\mathbb{Q}) \cong \mathbb{Q}$) and encodes the surgery obstruction for simply-connected 4-manifolds via $\sigma(M) \equiv 0 \pmod{16}$ (Rohlin's theorem).

In [ ]:
import numpy as np
from pysurgery.core.rational_surgery import (
    compute_l_group_rational, RationalObstruction,
    compute_l_group_p_local, PLocalObstruction,
    reconstruct_integral_obstruction,
    prime_local_obstruction_report, PrimeLocalReport,
)
from pysurgery.core.intersection_forms import IntersectionForm

print("=" * 70)
print("29 - Rational Surgery & L-Groups via CRT: Setup Complete")
print("=" * 70)

## Part 1: Why Integer L-Groups Are Hard

$L_0(\mathbb{Z}[\pi])$ classifies non-degenerate even symmetric bilinear forms over $\mathbb{Z}[\pi]$ up to stable isomorphism. Over $\mathbb{Q}$, this is completely determined by signature (Sylvester's law). Over $\mathbb{Z}_p$, it is determined by signature + Arf invariant. Over $\mathbb{Z}$, these local data are patched together by a global condition.

The CRT for quadratic forms (Hasse-Minkowski) says: two forms over $\mathbb{Q}$ are equivalent if and only if they are equivalent over $\mathbb{R}$ (same signature) and over $\mathbb{Q}_p$ for all primes $p$.

In [ ]:
# Hyperbolic form H = [[0,1],[1,0]] — the simplest non-trivial form
H_form = IntersectionForm(matrix=np.array([[0, 1], [1, 0]]))
print("Hyperbolic form H:")
print(f"  matrix:      {H_form.matrix.tolist()}")
print(f"  signature:   {H_form.signature()}")
print(f"  is_even:     {H_form.is_even}")
print(f"  is_unimodular: {H_form.is_unimodular}")
print()

# E₈ form — definite, even, rank 8
E8 = IntersectionForm(matrix=np.array([
    [-2, 1, 0, 0, 0, 0, 0, 0],
    [ 1,-2, 1, 0, 0, 0, 0, 0],
    [ 0, 1,-2, 1, 0, 0, 0, 1],
    [ 0, 0, 1,-2, 1, 0, 0, 0],
    [ 0, 0, 0, 1,-2, 1, 0, 0],
    [ 0, 0, 0, 0, 1,-2, 1, 0],
    [ 0, 0, 0, 0, 0, 1,-2, 0],
    [ 0, 0, 1, 0, 0, 0, 0,-2],
]))
print("E₈ form:")
print(f"  rank:        {E8.rank}")
print(f"  signature:   {E8.signature()}")
print(f"  is_even:     {E8.is_even}")
print(f"  is_unimodular: {E8.is_unimodular}")

## Part 2: Rational L-Group Obstruction

`compute_l_group_rational` computes the Wall obstruction class in $L_n(\mathbb{Q}[\pi])$. For $\pi$ trivial, this is determined by the signature. For $\pi$ finite, it involves the Schur index of the rational group algebra.

In [ ]:
# 4-dimensional surgery with trivial fundamental group
for form, name in [(H_form, "Hyperbolic H"), (E8, "E₈")]:
    rat: RationalObstruction = compute_l_group_rational(
        dimension=4,
        pi="trivial",
        form=form,
    )
    print(f"{name}:")
    print(f"  rational_signature:    {rat.rational_signature}")
    print(f"  vanishes_rationally:   {rat.vanishes_rationally}")
    print(f"  obstruction_class:     {rat.obstruction_class}")
    print(f"  exact:                 {rat.exact}")
    print(f"  theorem_tag:           {rat.theorem_tag}")
    print()

# With non-trivial π₁ = Z
rat_Z: RationalObstruction = compute_l_group_rational(
    dimension=4, pi="Z", form=H_form
)
print(f"H with π₁=Z: vanishes_rationally={rat_Z.vanishes_rationally}")
print("  (Z acts on form by deck transformations)")

## Part 3: p-Local L-Group Obstructions

`compute_l_group_p_local` computes the obstruction in $L_n(\mathbb{Z}_{(p)}[\pi])$. The key new invariant at $p = 2$ is the **Arf invariant**: for a form over $\mathbb{Z}/2$, it detects whether the form is split or not.

In [ ]:
# p-local obstructions for the E₈ form
primes = [2, 3, 5, 7]
print("p-local L₀ obstructions for E₈:")
print(f"{'prime':>6} {'vanishes_p_locally':>20} {'arf_invariant':>15} {'exact':>8}")
print("-" * 55)

p_local_results = {}
for p in primes:
    obs: PLocalObstruction = compute_l_group_p_local(
        dimension=4, pi="trivial", prime_p=p, form=E8
    )
    p_local_results[p] = obs
    print(f"  p={p}:  {str(obs.vanishes_p_locally):>20} {str(obs.arf_invariant):>15} {str(obs.exact):>8}")

print()
print("E₈ has signature -8 ≠ 0 mod 16, so it cannot be realised by a 4-manifold")
print("with trivial π₁ (Rohlin's theorem).")

## Part 4: CRT Reconstruction

`reconstruct_integral_obstruction` patches the rational and $p$-local data together. The output is a list of integers encoding the element of $L_n(\mathbb{Z}[\pi])$. If the list is all zeros, surgery is possible.

In [ ]:
# CRT reconstruction for E₈ form
rat_E8 = compute_l_group_rational(dimension=4, pi="trivial", form=E8)
integral = reconstruct_integral_obstruction(rat_E8, p_local_results)
print(f"Integral L-group obstruction (E₈): {integral}")
print(f"All zeros? {all(x == 0 for x in integral)}")
print("(Should be non-zero — E₈ cannot be realised as ∂W for W a smooth 4-manifold)")
print()

# Contrast with hyperbolic form H (signature 0, should vanish)
rat_H = compute_l_group_rational(dimension=4, pi="trivial", form=H_form)
p_local_H = {p: compute_l_group_p_local(4, "trivial", p, H_form) for p in primes}
integral_H = reconstruct_integral_obstruction(rat_H, p_local_H)
print(f"Integral L-group obstruction (H):  {integral_H}")
print(f"All zeros? {all(x == 0 for x in integral_H)}")
print("(Hyperbolic form: surgery is always possible — H bounds a trivial cobordism)")

## Part 5: Full Obstruction Report

`prime_local_obstruction_report` generates a comprehensive multi-prime analysis with all obstructions collated.

In [ ]:
report_E8: PrimeLocalReport = prime_local_obstruction_report(
    dimension=4, pi="trivial", form=E8, primes=[2, 3, 5]
)

print("Full obstruction report for E₈:")
print(report_E8.summary())
print()
print(f"all_vanish: {report_E8.all_vanish}")
print(f"exact:      {report_E8.exact}")
print(f"theorem_tag:{report_E8.theorem_tag}")

## Summary Checklist

- [x] Understood why integer $L$-groups require the rational + $p$-local strategy
- [x] Used `compute_l_group_rational` and read `RationalObstruction.rational_signature`
- [x] Used `compute_l_group_p_local` with multiple primes and read `arf_invariant`
- [x] Reconstructed the integral obstruction with `reconstruct_integral_obstruction`
- [x] Generated a full multi-prime report with `prime_local_obstruction_report`

## Next Steps
- **NB 11**: Wall groups give the algebraic context for the $L$-groups used here
- **NB 28**: Handle decompositions produce the intersection forms passed to these functions
- **NB 34**: The capstone passes the full obstruction report as step 6 of the surgery pipeline